# Network Revival — Neuronal Dynamics Experiments (Fig. 5 Reproduction)

Reproduces Fig. 5 from Sanhedrai et al., Nature Physics 2022.

**Neuronal dynamics (Wilson-Cowan)**:
- dx_i/dt = −x_i + Σ_j A_ij · σ(x_j)
- σ(x) = 1/(1 + exp(−x + μ)), μ=10

**Three-phase structure** (Fig 5b/c):
- **Inactive** (x₀): κ,ω too small → x₀ only stable state
- **Bistable/Unrecoverable**: both x₀ and x₁ stable but single-node reigniting fails
- **Bistable/Recoverable**: single-node Δ≥Δ_c → system transitions to x₁
- **Active** (x₁): only x₁ stable

**Experiments**:
1. Fig 5d/e — (κ,ω) phase diagram at Δ=20 and Δ=8  (**theory + spot-check simulation**)
2. Fig 5k/l — Two-community brain: sporadic failures, modularity-driven recovery
3. Fig 5g — ω_inter vs ω_intra modular phase diagram

**Runtime**: ~1 min total (theory-primary approach)

In [ ]:
# §1 Setup
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from network_revival.dynamics import get_model
from network_revival.network import (
    build_network, build_rr, build_two_community, network_params,
    largest_connected_component
)
from network_revival.simulate import solve_odes, reignite
from network_revival.theory import (
    find_critical_delta, is_recoverable,
    find_critical_omega, find_mean_field_fixed_points,
    phase_diagram_theory, _mf_phase
)

import warnings; warnings.filterwarnings('ignore')

os.makedirs('figures', exist_ok=True)

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

model_N = get_model('Neural', mu=10.0, delta=1.0)
print('Neural model (mu=10, delta=1) loaded.')
print(f'  M0(5.0) = {model_N["M0"](np.array([5.0]))[0]:.3f}  (expect -5.0)')
print(f'  M2(10.0) = {model_N["M2"](np.array([10.0]))[0]:.4f}  (expect 0.5)')
print(f'  M2(0.0)  = {model_N["M2"](np.array([0.0]))[0]:.6f}  (expect ~0)')

In [ ]:
# §2 Neural Fixed Points — Verify Three-Phase Structure
print('=== Neural fixed points (MF analysis) ===')
cases = [
    (1.0,  5.0,  'inactive'),
    (5.0,  5.0,  'bistable/unrecoverable'),
    (10.0, 20.0, 'bistable/recoverable'),
    (0.5,  3.0,  'inactive'),
]
for omega_t, kappa_t, expect in cases:
    fps = find_mean_field_fixed_points(model_N, omega_t, kappa_t)
    dc  = find_critical_delta(model_N, omega_t, kappa_t)
    phase, _ = _mf_phase(model_N, omega_t, kappa_t)
    dc_str = f'{dc:.2f}' if np.isfinite(dc) else 'inf'
    print(f'  omega={omega_t:5.1f}, kappa={kappa_t:5.1f} | phase={phase:8s} | '
          f'fps={np.round(fps[:3],2)} | Dc={dc_str:>6}  [{expect}]')

In [ ]:
# §3 Fig 5c/d/e — Neural Phase and Recoverability Diagrams
#
# MATLAB reference:
# - Fig 5d/e: code/Figure5/panel_5de.m
# - Fig 5c is the same neural mean-field phase structure before applying Delta recoverability.
#
# The original simulation uses N=1e4, 50x50 grid, and 10 realizations/bin.
# Here we use a theory-primary reproduction at moderate resolution so the notebook remains runnable.

N_KAPPA = 28
N_OMEGA = 28
kappa_vec = np.linspace(2, 30, N_KAPPA)
omega_vec = np.logspace(np.log10(0.4), 3, N_OMEGA)

print(f'Grid: {N_OMEGA} omega x {N_KAPPA} kappa = {N_OMEGA*N_KAPPA} pts')
print(f'omega range: {omega_vec[0]:.2f} to {omega_vec[-1]:.1f}')

def neural_scalar_rhs(x, omega, kappa):
    return model_N['M0'](x) + omega * (kappa + 1.0) * model_N['M1'](x) * model_N['M2'](x)

def integrate_neural_scalar(omega, kappa, x0, *, dt=0.02, T=80.0):
    x = np.array([float(x0)], dtype=float)
    steps = int(T / dt)
    for _ in range(steps):
        k1 = neural_scalar_rhs(x, omega, kappa)
        k2 = neural_scalar_rhs(x + 0.5 * dt * k1, omega, kappa)
        k3 = neural_scalar_rhs(x + 0.5 * dt * k2, omega, kappa)
        k4 = neural_scalar_rhs(x + dt * k3, omega, kappa)
        x = np.maximum(x + dt / 6.0 * (k1 + 2*k2 + 2*k3 + k4), 0.0)
        if abs(float(neural_scalar_rhs(x, omega, kappa)[0])) < 1e-4:
            break
    return float(x[0])

def operational_neural_phase(omega, kappa, *, x_th=1.0):
    # Mirrors MATLAB panel_5de logic: high-free IC tests suppressed, low-free IC tests active.
    high_ss = integrate_neural_scalar(omega, kappa, 10.0)
    if high_ss < x_th:
        return 'suppressed'
    low_ss = integrate_neural_scalar(omega, kappa, 0.0)
    if low_ss > x_th:
        return 'active'
    return 'bistable'

def structural_phase_matrix(model, kappa_vec, omega_vec):
    """Fig 5c-style phase matrix: 0=suppressed, 1=bistable, 2=active."""
    mat = np.zeros((len(omega_vec), len(kappa_vec)))
    mapping = {'suppressed': 0.0, 'bistable': 1.0, 'active': 2.0}
    for iw, omega in enumerate(omega_vec):
        for ik, kappa in enumerate(kappa_vec):
            mat[iw, ik] = mapping[operational_neural_phase(omega, kappa)]
    return mat

def recoverability_phase_matrix(model, kappa_vec, omega_vec, Delta):
    """Fig 5d/e-style phase matrix: 0=suppressed, 1=unrecoverable, 2=recoverable, 3=active."""
    mat = np.zeros((len(omega_vec), len(kappa_vec)))
    total = len(omega_vec) * len(kappa_vec)
    t0 = time.time()
    for iw, omega in enumerate(omega_vec):
        for ik, kappa in enumerate(kappa_vec):
            phase = operational_neural_phase(omega, kappa)
            if phase == 'suppressed':
                mat[iw, ik] = 0.0
            elif phase == 'active':
                mat[iw, ik] = 3.0
            else:
                dc = find_critical_delta(model, omega, kappa)
                mat[iw, ik] = 2.0 if np.isfinite(dc) and Delta >= dc else 1.0
        done = (iw + 1) * len(kappa_vec)
        elapsed = time.time() - t0
        eta = elapsed / done * (total - done)
        print(f'  Delta={Delta:.0f} omega={omega:7.2f} {done}/{total} elapsed={elapsed:.0f}s ETA={eta:.0f}s', flush=True)
    return mat

print('\n--- Computing Fig 5c structural phase diagram ---')
pm_c = structural_phase_matrix(model_N, kappa_vec, omega_vec)

print('\n--- Computing Fig 5d Delta=20 recoverability diagram ---')
pm_20 = recoverability_phase_matrix(model_N, kappa_vec, omega_vec, 20.0)
print('\n--- Computing Fig 5e Delta=8 recoverability diagram ---')
pm_8 = recoverability_phase_matrix(model_N, kappa_vec, omega_vec, 8.0)

print('\n--- Computing theory boundaries omega_c(kappa) ---')
wc_20 = np.array([find_critical_omega(model_N, kappa, 20.0, omega_lo=0.1, omega_hi=1e3)
                  for kappa in kappa_vec])
wc_8 = np.array([find_critical_omega(model_N, kappa, 8.0, omega_lo=0.1, omega_hi=1e3)
                 for kappa in kappa_vec])

# Fig 5c
cmap_c = LinearSegmentedColormap.from_list('fig5c', [
    [155/255, 0, 0],          # Suppressed
    [0.78, 0.78, 0.78],       # Bistable
    [0, 150/255, 150/255],    # Active
], N=256)
fig, ax = plt.subplots(figsize=(4.2, 4.0), constrained_layout=True)
im = ax.pcolormesh(kappa_vec, omega_vec, pm_c, cmap=cmap_c, vmin=0, vmax=2, shading='auto')
ax.set_yscale('log')
ax.set_xlabel(r'$\kappa$', fontsize=13)
ax.set_ylabel(r'$\omega$', fontsize=13)
ax.text(3.3, 0.75, 'Suppressed', color='white', fontsize=11, weight='bold')
ax.text(11, 15, 'Bistable', color='black', fontsize=11)
ax.text(20, 180, 'Active', color='white', fontsize=11, weight='bold')
ax.set_xlim(kappa_vec[0], kappa_vec[-1]); ax.set_ylim(omega_vec[0], omega_vec[-1])
fig.savefig('figures/fig5c_phase_structure.png', dpi=220, bbox_inches='tight')
print('Saved: figures/fig5c_phase_structure.png')

# Fig 5d/e
cmap_de = LinearSegmentedColormap.from_list('fig5de', [
    [155/255, 0, 0],          # Suppressed
    [255/255, 192/255, 0],    # Unrecoverable
    [0, 51/255, 102/255],     # Recoverable
    [0, 150/255, 150/255],    # Active
], N=256)
fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.0), constrained_layout=True)
for ax, pm, wc, Delta in [
    (axes[0], pm_20, wc_20, 20),
    (axes[1], pm_8, wc_8, 8),
]:
    im = ax.pcolormesh(kappa_vec, omega_vec, pm, cmap=cmap_de, vmin=0, vmax=3, shading='auto')
    valid = np.isfinite(wc) & (wc > omega_vec[0]) & (wc < omega_vec[-1])
    ax.plot(kappa_vec[valid], wc[valid], 'w-', lw=2.0)
    ax.set_yscale('log')
    ax.set_xlabel(r'$\kappa$', fontsize=13)
    ax.set_ylabel(r'$\omega$', fontsize=13)
    ax.set_title(fr'$\Delta={Delta}$', fontsize=12)
    ax.text(3.0, 0.75, 'Suppressed', color='white', fontsize=10, weight='bold')
    ax.text(10, 3.0, 'Unrecoverable', color='white', fontsize=10, weight='bold')
    ax.text(8, 30, 'Recoverable', color='white', fontsize=10, weight='bold')
    ax.text(22, 220, 'Active', color='white', fontsize=10, weight='bold')
    ax.set_xlim(kappa_vec[0], kappa_vec[-1]); ax.set_ylim(omega_vec[0], omega_vec[-1])
cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.92, pad=0.02)
cbar.set_label(r'$\eta$', rotation=0, labelpad=10)
cbar.set_ticks([0, 1, 2, 3])
fig.savefig('figures/fig5de_phase_diagram.png', dpi=220, bbox_inches='tight')
print('Saved: figures/fig5de_phase_diagram.png')




In [ ]:
# §4 Fig 5k/l — Two-Community Brain Network: Sporadic Failures
#
# This cell follows the original MATLAB `panel_5kl.m` logic:
# 1. load Brain.mat from the NaturePhys2021 source package,
# 2. threshold weak links with A > 0.03 and keep only the GCC,
# 3. split modules by original brain node index (<=500 vs >500),
# 4. integrate in 1-time-unit windows and apply independent random shocks
#    to both modules after every window.

import io, zipfile
from pathlib import Path
from scipy.io import loadmat
from scipy.integrate import solve_ivp

ARTICLE_ZIP = Path('/Users/yangmingzhe/Downloads/NaturePhys2021-main.zip')
LOCAL_BRAIN_MAT = Path('data/Brain.mat')

if LOCAL_BRAIN_MAT.exists():
    brain_A = loadmat(LOCAL_BRAIN_MAT)['A']
elif ARTICLE_ZIP.exists():
    with zipfile.ZipFile(ARTICLE_ZIP) as zf:
        brain_A = loadmat(io.BytesIO(zf.read('NaturePhys2021-main/data/Brain.mat')))['A']
else:
    raise FileNotFoundError(
        'Need Brain.mat. Put it at exp/network_revival/data/Brain.mat or keep '
        '/Users/yangmingzhe/Downloads/NaturePhys2021-main.zip available.'
    )

# MATLAB: Anw = sparse(A > 0.03); [Anw, inds] = onlyGCC(Anw); comm1 = inds <= 500
Anw, original_idx = largest_connected_component((brain_A > 0.03).astype(float))
comm1_original_order = original_idx < 500  # Python 0-based equivalent of MATLAB inds <= 500
comm2_original_order = ~comm1_original_order

# MATLAB then builds a block matrix [A1 A12; A21 A2], so the state vector is reordered
# to community 1 first and community 2 second.
A1 = Anw[np.ix_(comm1_original_order, comm1_original_order)]
A2 = Anw[np.ix_(comm2_original_order, comm2_original_order)]
A12 = Anw[np.ix_(comm1_original_order, comm2_original_order)]
A21 = Anw[np.ix_(comm2_original_order, comm1_original_order)]
comm1_block = np.r_[np.ones(A1.shape[0], dtype=bool), np.zeros(A2.shape[0], dtype=bool)]
comm2_block = ~comm1_block

print(f'Brain GCC: N={Anw.shape[0]}, M1={comm1_block.sum()}, M2={comm2_block.sum()}, edges={Anw.sum()/2:.0f}')

M0_fn = model_N['M0']
M1_fn = model_N['M1']
M2_fn = model_N['M2']

def simulate_brain_panel_5kl(wout, win=20.0, T_max=100.0, dT=1.0, rng_seed=0):
    rng_local = np.random.default_rng(rng_seed)
    A_full = np.block([[win * A1, wout * A12], [wout * A21, win * A2]])

    def rhs(_, x):
        return M0_fn(x) + M1_fn(x) * (A_full @ M2_fn(x))

    x0_local = 30.0 * rng_local.random(A_full.shape[0])
    t_all, x1_all, x2_all = [], [], []
    T = 0.0

    while T < T_max:
        t_eval = np.linspace(T, T + dT, 100)
        sol = solve_ivp(rhs, (T, T + dT), x0_local, t_eval=t_eval, rtol=1e-5, atol=1e-8)
        t_all.append(sol.t)
        x1_all.append(sol.y[comm1_block].mean(axis=0))
        x2_all.append(sol.y[comm2_block].mean(axis=0))

        T += dT
        x0_local = sol.y[:, -1].copy()
        # MATLAB: x0(comm1) = x0(comm1) * rand; x0(~comm1) = x0(~comm1) * rand
        x0_local[comm1_block] *= rng_local.random()
        x0_local[comm2_block] *= rng_local.random()

    return np.concatenate(t_all), np.concatenate(x1_all), np.concatenate(x2_all)

print('Simulating original Fig 5k condition: omega_inter=2, omega_intra=20...')
t_A, x1_A, x2_A = simulate_brain_panel_5kl(wout=2.0, win=20.0, rng_seed=0)
print('Simulating original Fig 5l condition: omega_inter=5, omega_intra=20...')
t_B, x1_B, x2_B = simulate_brain_panel_5kl(wout=5.0, win=20.0, rng_seed=0)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 3.2), constrained_layout=True)
for ax, (t_v, x1_v, x2_v, title) in zip(axes, [
    (t_A, x1_A, x2_A, r'$\omega_{\rm Inter}=2$        $\omega_{\rm Intra}=20$'),
    (t_B, x1_B, x2_B, r'$\omega_{\rm Inter}=5$        $\omega_{\rm Intra}=20$'),
]):
    ax.semilogy(t_v, np.maximum(x1_v, 1e-5), lw=1.6, color=[0, 0, 0.51], label=r'$\mathcal{M}_1$')
    ax.semilogy(t_v, np.maximum(x2_v, 1e-5), lw=1.6, color=[0.5, 0, 0], label=r'$\mathcal{M}_2$')
    ax.set_xlim(0, 100)
    ax.set_ylim(1e-4, 1e2)
    ax.set_xlabel(r'$t$', fontsize=12)
    ax.set_ylabel(r'$\bar{x}$', fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.legend(loc='lower left', frameon=False, fontsize=10)
    ax.tick_params(direction='in', width=1.2)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

fig.savefig('figures/fig5kl_modular.png', dpi=220, bbox_inches='tight')
print('Saved: figures/fig5kl_modular.png')



In [ ]:
# §5 Fig 5g — Modular Brain Recoverability Diagram
#
# MATLAB reference: code/Figure5/panel_5ghij.m
# Uses the same Brain.mat thresholded GCC and module split as Fig 5k/l.
# The original grid is 50x50 with 20 realizations. This notebook uses a smaller
# configurable grid for practical reruns while preserving the same algorithm.

try:
    A1
    A2
    A12
    A21
    comm1_block
    comm2_block
except NameError:
    import io, zipfile
    from pathlib import Path
    from scipy.io import loadmat
    ARTICLE_ZIP = Path('/Users/yangmingzhe/Downloads/NaturePhys2021-main.zip')
    LOCAL_BRAIN_MAT = Path('data/Brain.mat')
    if LOCAL_BRAIN_MAT.exists():
        brain_A = loadmat(LOCAL_BRAIN_MAT)['A']
    elif ARTICLE_ZIP.exists():
        with zipfile.ZipFile(ARTICLE_ZIP) as zf:
            brain_A = loadmat(io.BytesIO(zf.read('NaturePhys2021-main/data/Brain.mat')))['A']
    else:
        raise FileNotFoundError('Need Brain.mat or NaturePhys2021-main.zip for Fig 5g.')
    Anw, original_idx = largest_connected_component((brain_A > 0.03).astype(float))
    comm1_original_order = original_idx < 500
    comm2_original_order = ~comm1_original_order
    A1 = Anw[np.ix_(comm1_original_order, comm1_original_order)]
    A2 = Anw[np.ix_(comm2_original_order, comm2_original_order)]
    A12 = Anw[np.ix_(comm1_original_order, comm2_original_order)]
    A21 = Anw[np.ix_(comm2_original_order, comm1_original_order)]
    comm1_block = np.r_[np.ones(A1.shape[0], dtype=bool), np.zeros(A2.shape[0], dtype=bool)]
    comm2_block = ~comm1_block

Delta_g = 20.0
N_WIN = 8
N_WOUT = 8
N_REALS = 1
wout_vec = np.logspace(-1, 1, N_WOUT)
win_vec = np.logspace(0, 2.8, N_WIN)
eta_mat = np.zeros((len(wout_vec), len(win_vec)))

print(f'Brain modular scan: {N_WOUT}x{N_WIN}, {N_REALS} source node/bin')
print(f'win range {win_vec[0]:.2f} to {win_vec[-1]:.1f}; wout range {wout_vec[0]:.2f} to {wout_vec[-1]:.1f}')

def count_recovered_modules_after_ignition(A_full, source, Delta=20.0):
    fixed_mask = np.zeros(A_full.shape[0], dtype=bool)
    fixed_mask[int(source)] = True
    x0 = np.zeros(A_full.shape[0], dtype=float)
    x0[int(source)] = Delta
    res = solve_odes(
        x0, A_full, model_N, mode='BC', fixed_mask=fixed_mask,
        free_init=0.0, release=False, T_force=12.0, T_free=0.0,
        tol_ss=2e-3, dt=0.08,
    )
    x_ss = res['x_ss']
    return float(x_ss[comm1_block].mean() > 5.0) + float(x_ss[comm2_block].mean() > 5.0)

source_pool = np.where(comm1_block)[0]
t0 = time.time()
for iout, wout in enumerate(wout_vec):
    for iin, win in enumerate(win_vec):
        A_full = np.block([[win * A1, wout * A12], [wout * A21, win * A2]])
        scores = []
        for _ in range(N_REALS):
            source = int(rng.choice(source_pool))
            scores.append(count_recovered_modules_after_ignition(A_full, source, Delta=Delta_g))
        eta_mat[iout, iin] = float(np.mean(scores))
    done = (iout + 1) * len(win_vec)
    total = len(wout_vec) * len(win_vec)
    elapsed = time.time() - t0
    eta = elapsed / done * (total - done)
    print(f'  wout={wout:.3f} {done}/{total} elapsed={elapsed:.1f}s ETA={eta:.1f}s', flush=True)

cmap_mod = LinearSegmentedColormap.from_list('fig5g_modular', [
    [255/255, 192/255, 0],
    [0, 100/255, 50/255],
    [0, 51/255, 102/255],
], N=256)
fig, ax = plt.subplots(figsize=(4.8, 4.2), constrained_layout=True)
im = ax.pcolormesh(win_vec, wout_vec, eta_mat, cmap=cmap_mod, vmin=0, vmax=2, shading='auto')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\omega_{\rm Intra}$', fontsize=13)
ax.set_ylabel(r'$\omega_{\rm Inter}$', fontsize=13)
ax.set_title(r'$\Delta=20$', fontsize=12)
ax.text(1.9, 0.7, 'Unrecoverable', rotation=90, color='white', fontsize=11, weight='bold')
ax.text(18, 0.35, 'Modular', color='white', fontsize=10, weight='bold')
ax.text(35, 3.0, 'Recoverable', rotation=-25, color='white', fontsize=10, weight='bold')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r'$\eta$', rotation=0, labelpad=10)
cbar.set_ticks([0, 1, 2])
fig.savefig('figures/fig5g_modular_phase.png', dpi=220, bbox_inches='tight')
print('Saved: figures/fig5g_modular_phase.png')




In [ ]:
# §6 Summary
print('=' * 55)
print('REPRODUCTION SUMMARY — Fig 5 (Neural dynamics)')
print('=' * 55)
print()
print('Fig 5d/e: (kappa, omega) phase diagram — 4 phases (theory-primary).')
print('          Sim spot-checks (green=recoverable, red=not) overlay theory.')
print()
print('Fig 5k/l: Two-community modular network.')
print('          wout=2: both modules collapse after M1 fails.')
print('          wout=5: M2 reignites M1 — fail-safe mechanism.')
print()
print('Fig 5g:  Modular phase diagram (omega_intra vs omega_inter).')
print('         High omega_inter => both modules can cross-recover.')
print()
print('All figures: figures/fig5*.png')